In [ ]:
"""data science is an interdisciplinary field that uses scientific methods
processes algorithms and systems to extract knowledge and insights from noisy
structured and unstructured data""".

In [ ]:
We extract overlapping 3-word sequences:

    "data science is" → Input: ["data", "science"], Target: "is"
    "science is an" → Input: ["science", "is"], Target: "an"
    "is an interdisciplinary" → Input: ["is", "an"], Target: "interdisciplinary"

The model sees many such examples and learns patterns like:

    "data science" → "is"
    "scientific methods" → "processes"

In [ ]:
Once trained, we can feed a seed phrase to the model, like:

    "data science"

The model embeds these words → feeds them into LSTM → produces a probability distribution over all vocabulary words for the next word.

It picks the word with the highest probability, say "is".

Then we slide the window:

    "science is" → predict "an"
    "is an" → predict "interdisciplinary"
    …and so on.

This is called auto-regressive generation — we keep feeding the model its own outputs.

In [ ]:
It won’t generalize to topics not in the training data.
It's not like GPT or BERT — it's a shallow LSTM trained on a small dataset.
It doesn't understand meaning — only word co-occurrence patterns.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from sklearn.preprocessing import OneHotEncoder

# Sample corpus (keep it small for demo purposes)
corpus = """data science is an interdisciplinary field that uses scientific methods
processes algorithms and systems to extract knowledge and insights from noisy
structured and unstructured data""".lower().split()

# Build vocabulary
word2idx = defaultdict(lambda: len(word2idx))
idx2word = dict()

# Fix vocabulary
vocab = sorted(set(corpus))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

# Prepare sequences: input (2 words) → target (next word)
sequences = []
for i in range(2, len(corpus)):
    seq = corpus[i-2:i+1]  # 2 words + target word
    sequences.append(seq)

# Convert to indices
def encode(seq):
    return [word2idx[w] for w in seq]

encoded_sequences = [encode(seq) for seq in sequences]

# Separate input and target
X = [seq[:-1] for seq in encoded_sequences]
y = [seq[-1] for seq in encoded_sequences]

X = torch.tensor(X)
y = torch.tensor(y)

# Dataset & Dataloader
class WordDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = WordDataset(X, y)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Define model
class WordPredictor(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(WordPredictor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)               # (batch, seq_len, embed_dim)
        out, _ = self.lstm(x)               # (batch, seq_len, hidden_dim)
        out = self.fc(out[:, -1, :])        # last time step output
        return out

# Initialize model
embed_dim = 10
hidden_dim = 128
model = WordPredictor(vocab_size, embed_dim, hidden_dim)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Train
for epoch in range(300):
    for xb, yb in loader:
        out = model(xb)
        loss = criterion(out, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Function to predict next word
def predict_next_word(model, seed_text):
    model.eval()
    with torch.no_grad():
        tokens = [word2idx.get(w, 0) for w in seed_text.lower().split()]
        tokens = tokens[-2:]  # take last 2 tokens
        if len(tokens) < 2:
            tokens = [0] * (2 - len(tokens)) + tokens
        input_tensor = torch.tensor(tokens).unsqueeze(0)
        output = model(input_tensor)
        pred_idx = torch.argmax(output, dim=1).item()
        return idx2word[pred_idx]

# Generate new text
seed = "data science"
print(f"Seed: {seed}")
for _ in range(10):
    next_word = predict_next_word(model, seed)
    seed += ' ' + next_word
print("Generated text:", seed)


Epoch 0, Loss: 3.2438
Epoch 50, Loss: 0.0004
Epoch 100, Loss: 0.0003
Epoch 150, Loss: 0.0001
Epoch 200, Loss: 0.0001
Epoch 250, Loss: 0.0000
Seed: data science
Generated text: data science is an interdisciplinary field that uses scientific methods processes algorithms


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

# New paragraph as training corpus
text = """
machine learning is a field of artificial intelligence that focuses on teaching machines to learn from data.
deep learning is a subfield of machine learning that uses neural networks.
data scientists often use python and tools like pandas and scikit learn.
artificial intelligence is transforming industries like healthcare, finance, and education.
""".lower().split()

# Build vocab
word_list = sorted(set(text))
word2idx = {w: i for i, w in enumerate(word_list)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(word_list)
.
# Generate sequences (2 words → next word)
sequences = []
for i in range(2, len(text)):
    seq = text[i-2:i+1]
    if all(w in word2idx for w in seq):  # ensure all in vocab
        sequences.append([word2idx[w] for w in seq])

# Separate input and target
X = torch.tensor([s[:-1] for s in sequences])
y = torch.tensor([s[-1] for s in sequences])

# Dataset
class WordDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = WordDataset(X, y)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# LSTM Model
class WordPredictor(nn.Module):
    def __init__(self, vocab_size, embed_dim=10, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embed(x)                 # (batch, 2, embed_dim)
        out, _ = self.lstm(x)             # (batch, 2, hidden)
        out = self.fc(out[:, -1, :])      # (batch, vocab_size)
        return out

model = WordPredictor(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Training loop
for epoch in range(300):
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

# Prediction
def predict_next_word(model, seed_text):
    model.eval()
    words = seed_text.lower().split()
    if len(words) < 2:
        words = ["<pad>"] * (2 - len(words)) + words
    words = words[-2:]
    input_ids = torch.tensor([[word2idx.get(w, 0) for w in words]])
    with torch.no_grad():
        output = model(input_ids)
        pred_idx = torch.argmax(output, dim=1).item()
        return idx2word[pred_idx]

# Generate sentence
seed = "machine learning"
print(f"\nSeed: {seed}")
for _ in range(12):
    next_word = predict_next_word(model, seed)
    seed += " " + next_word
print("Generated text:", seed)


Epoch 0: Loss = 3.8698
Epoch 50: Loss = 0.0005
Epoch 100: Loss = 0.0003
Epoch 150: Loss = 0.0001
Epoch 200: Loss = 0.0000
Epoch 250: Loss = 0.0000

Seed: machine learning
Generated text: machine learning is a subfield of machine learning is a subfield of machine learning


In [ ]:
# GRU

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Training corpus (multi-sentence paragraph)
text = """
machine learning is a field of artificial intelligence that focuses on teaching machines to learn from data.
deep learning is a subfield of machine learning that uses neural networks.
data scientists often use python and tools like pandas and scikit learn.
artificial intelligence is transforming industries like healthcare, finance, and education.
""".lower().split()

# Build vocab
word_list = sorted(set(text))
word2idx = {w: i for i, w in enumerate(word_list)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(word_list)

# Generate sequences (2-word input → 1-word target)
sequences = []
for i in range(2, len(text)):
    seq = text[i-2:i+1]
    if all(w in word2idx for w in seq):
        sequences.append([word2idx[w] for w in seq])

# Prepare inputs and targets
X = torch.tensor([s[:-1] for s in sequences])
y = torch.tensor([s[-1] for s in sequences])

# Dataset and Dataloader
class WordDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = WordDataset(X, y)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# 🔄 GRU-based model
class GRUWordPredictor(nn.Module):
    def __init__(self, vocab_size, embed_dim=10, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)              # (batch, seq_len, embed_dim)
        out, _ = self.gru(x)               # (batch, seq_len, hidden_dim)
        out = self.fc(out[:, -1, :])       # (batch, vocab_size)
        return out

# Initialize model, loss, optimizer
model = GRUWordPredictor(vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Train the model
for epoch in range(300):
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

# Inference function
def predict_next_word(model, seed_text):
    model.eval()
    words = seed_text.lower().split()
    words = words[-2:] if len(words) >= 2 else ["<pad>"] * (2 - len(words)) + words
    tokens = torch.tensor([[word2idx.get(w, 0) for w in words]])
    with torch.no_grad():
        out = model(tokens)
        pred_idx = torch.argmax(out, dim=1).item() #Applies argmax() to get the predicted word index
        return idx2word[pred_idx]




# Generate sentence from seed
seed = "machine learning"
print(f"\nSeed: {seed}")
for _ in range(12):
    next_word = predict_next_word(model, seed)
    seed += " " + next_word
print("Generated text:", seed)


Epoch 0: Loss = 3.9361
Epoch 50: Loss = 0.0023
Epoch 100: Loss = 0.0001
Epoch 150: Loss = 0.0001
Epoch 200: Loss = 0.0001
Epoch 250: Loss = 0.0000

Seed: machine learning
Generated text: machine learning is a field of artificial intelligence is transforming industries like healthcare, finance,


In [ ]:
# Corpus Preparation

🔹 We're using a multi-sentence paragraph as training data.
🔹 .lower() makes the words lowercase (standard NLP preprocessing).
🔹 .split() splits by whitespace → gives a list of words (i.e., word-level tokenization).

In [ ]:
# Building the Vocabulary

# word_list = sorted(set(text))
# word2idx = {w: i for i, w in enumerate(word_list)}
# idx2word = {i: w for w, i in word2idx.items()}

🔹 word_list contains all unique words in the text.
🔹 word2idx maps word → integer index (e.g., 'machine': 17)
🔹 idx2word is the reverse: index → word (needed for generating output)


This is the vocabulary — the model will only predict words from this set.

In [ ]:
# Sequence Creation

# for i in range(2, len(text)):
#     seq = text[i-2:i+1]
#     ...

🔹 We create sliding 3-word windows: [word1, word2, target_word]
        → Example: ["machine", "learning", "is"]

🔹 Each sequence becomes a training example:

    Input: first 2 words → "machine learning"
    Target: 3rd word → "is"

In [ ]:
# Encoding Sequences

sequences.append([word2idx[w] for w in seq])

🔹 Each word is replaced with its index (from word2idx).
🔹 Now we have integer-based sequences → suitable for training the model.

In [ ]:
# Separating Inputs and Targets

X = torch.tensor([s[:-1] for s in sequences])  # input: 2-word indices It returns all elements of the list except the last one.
y = torch.tensor([s[-1] for s in sequences])   # target: next word index

🔹 X contains all 2-word input sequences
🔹 y contains the correct next word to be predicted

# What Is an Embedding Layer?

In [ ]:
The embedding layer maps discrete word indices into dense, continuous vectors — capturing semantic similarity between words.
Why do we need it?

Let’s say your vocabulary is:

word2idx = {'machine': 0, 'learning': 1, 'is': 2, 'a': 3}

Using these integers directly as input (e.g., 0, 1, 2...) is meaningless to a neural network
— there's no mathematical relationship between 'machine' (0) and 'learning' (1).

    ********** Enter: Embeddings — they assign a learnable vector to each word.******************

In [ ]:
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)

For example:

nn.Embedding(50, 10)

This creates a lookup table of shape (50, 10):

    50 → vocabulary size (number of words)
    10 → each word is represented by a 10-dimensional vector

In [ ]:
How Embedding Learns

During backpropagation:
    If 'learning' in a specific context leads to a lower loss,
    The embedding vector for 'learning' is updated accordingly,
    So next time, the model can better use that context.

The embedding matrix is just a torch.nn.Parameter (i.e., weights) like any other layer.

In [ ]:
import torch
import torch.nn as nn

vocab_size = 5
embed_dim = 4
embedding = nn.Embedding(vocab_size, embed_dim)

input_ids = torch.tensor([[0, 1, 2], [3, 4, 1]])  # shape: (batch=2, seq_len=3)
embedded = embedding(input_ids)

print("Input shape:", input_ids.shape)
print("Embedded shape:", embedded.shape)  # (2, 3, 4)


Input shape: torch.Size([2, 3])
Embedded shape: torch.Size([2, 3, 4])


# Visualizing Embedding Lookup

In [ ]:
# Vocabulary

word2idx = {'machine': 0, 'learning': 1, 'is': 2, 'fun': 3}


In [ ]:
import torch
import torch.nn as nn

# Define vocab size and embedding dimension
vocab_size = 4
embed_dim = 3

# Create embedding layer
embedding = nn.Embedding(vocab_size, embed_dim)

# Let's inspect the initial embedding weights
print("Embedding Matrix (Before any training):")
print(embedding.weight)


Embedding Matrix (Before any training):
Parameter containing:
tensor([[ 0.8215,  0.7082, -0.2864],
        [-1.2630, -0.5726, -0.3567],
        [ 0.6517,  0.6316, -1.9574],
        [-0.8165,  3.1347, -1.5060]], requires_grad=True)


In [ ]:
tensor([[ 1.9258, -0.9333,  0.6522],  # for 'machine' (index 0)
        [-0.1795,  0.1284,  1.5508], # for 'learning' (index 1)
        [ 0.9511,  1.2348,  0.4296],  # for 'is' (index 2)
        [-2.3161, -1.1541, -2.4551]] # for 'fun' (index 3)

In [ ]:
# lookup specific words

# Simulate a sentence: "machine learning is"
word_indices = torch.tensor([[0, 1, 2]])  # shape: (1, 3)
embedded = embedding(word_indices)

print("Input Word Indices:", word_indices)
print("Corresponding Embeddings:")
print(embedded)


Input Word Indices: tensor([[0, 1, 2]])
Corresponding Embeddings:
tensor([[[ 1.9258, -0.9333,  0.6522],
         [-0.1795,  0.1284,  1.5508],
         [ 0.9511,  1.2348,  0.4296]]], grad_fn=<EmbeddingBackward0>)


In [ ]:
#before aand after training

import torch
import torch.nn as nn
import torch.optim as optim


word2idx = {'machine': 0, 'learning': 1}
idx2word = {v: k for k, v in word2idx.items()}
vocab_size = len(word2idx)
embed_dim = 4

# Create Embedding + Simple Classifier
embedding = nn.Embedding(vocab_size, embed_dim)
linear = nn.Linear(embed_dim, 1)  # binary classifier

model = nn.Sequential(embedding, linear)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training data (predict if word is 'learning')
X = torch.tensor([0, 1])        # word indices: 'machine' and 'learning'
y = torch.tensor([0.0, 1.0])    # label: 0 if machine, 1 if learning

# See initial embedding for 'machine'
print("Initial embedding for 'machine':")
print(embedding.weight[0].detach().clone())


for epoch in range(20):
    optimizer.zero_grad()
    output = model(X)
    loss = loss_fn(output.squeeze(), y)
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# Step 6: See updated embedding for 'machine'
print("Updated embedding for 'machine':")
print(embedding.weight[0].detach())


Initial embedding for 'machine':
tensor([-0.2826, -1.0257,  0.3104, -0.4879])
Epoch 0 | Loss: 0.6583
Epoch 5 | Loss: 0.5318
Epoch 10 | Loss: 0.4342
Epoch 15 | Loss: 0.3551
Updated embedding for 'machine':
tensor([-0.2386, -0.9097,  0.1639, -0.6989])


# character level prediction

In [ ]:
import torch
import torch.nn as nn

# Data
text = "hello"
chars = sorted(set(text))  # ['e', 'h', 'l', 'o']
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(chars)

# Input: 'h', 'e', 'l', 'l' → Output: 'o'
input_seq = [char2idx[c] for c in text[:-1]]  # 'hell' → [1, 0, 2, 2]
target = char2idx[text[-1]]  # 'o'

X = torch.tensor([input_seq])  # shape (1, 4)
y = torch.tensor([target])     # shape (1)

# Model
class SimpleCharGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=8, hidden_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embed(x)  # (B, T, D)
        out, _ = self.gru(x)  # (B, T, H)
        out = self.fc(out[:, -1, :])  # last time step
        return out

model = SimpleCharGRU(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Train
for epoch in range(300):
    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        pred_idx = torch.argmax(pred, dim=1).item()
        pred_char = idx2char[pred_idx]
        print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Predicted: {pred_char}")

# Predict from "hell"
def predict(model, input_str):
    model.eval()
    input_ids = torch.tensor([[char2idx[c] for c in input_str]])
    with torch.no_grad():
        out = model(input_ids)
        next_idx = torch.argmax(out, dim=1).item()
        return idx2char[next_idx]

result = predict(model, "hell")
print("Prediction for 'hell' →", result)


Epoch 0 | Loss: 1.1419 | Predicted: o
Epoch 50 | Loss: 0.0017 | Predicted: o
Epoch 100 | Loss: 0.0009 | Predicted: o
Epoch 150 | Loss: 0.0006 | Predicted: o
Epoch 200 | Loss: 0.0005 | Predicted: o
Epoch 250 | Loss: 0.0004 | Predicted: o

🧠 Prediction for 'hell' → o


In [ ]:
# Extended character level prediction

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Raw text
text = "hello how are you"
chars = sorted(set(text))  # unique characters
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for ch, i in char2idx.items()}
vocab_size = len(chars)

# Encode entire text
encoded = [char2idx[ch] for ch in text]

# Create (X, y) pairs with sequence length = 4
seq_length = 4
sequences = []
for i in range(len(encoded) - seq_length):
    X_seq = encoded[i:i+seq_length]
    y_seq = encoded[i+seq_length]
    sequences.append((X_seq, y_seq))

X = torch.tensor([s[0] for s in sequences])
y = torch.tensor([s[1] for s in sequences])


In [ ]:
class CharDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = CharDataset(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)


In [ ]:
class CharGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=8, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embed(x)  # (B, T) → (B, T, embed_dim)
        out, _ = self.gru(x)  # (B, T, hidden)
        out = self.fc(out[:, -1, :])  # take output at last timestep
        return out


In [ ]:
model = CharGRU(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(300):
    for xb, yb in loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 2.8323
Epoch 50, Loss: 0.0005
Epoch 100, Loss: 0.0002
Epoch 150, Loss: 0.0002
Epoch 200, Loss: 0.0001
Epoch 250, Loss: 0.0001


In [ ]:
def predict_next_chars(model, seed_text, num_chars=10):
    model.eval()
    input_seq = [char2idx[c] for c in seed_text]

    for _ in range(num_chars):
        x = torch.tensor([input_seq[-seq_length:]])  # shape: (1, 4)
        with torch.no_grad():
            pred = model(x)
            next_idx = torch.argmax(pred, dim=1).item()
        input_seq.append(next_idx)

    result = ''.join(idx2char[i] for i in input_seq)
    return result


In [ ]:
seed = "hell"
generated = predict_next_chars(model, seed, num_chars=10)
print("Generated:", generated)


Generated: hello how are 


In [ ]:
# Types

| Type                        | Input                   | Output               | Examples                                       |
| --------------------------- | ----------------------- | -------------------- | ---------------------------------------------- |
| 1. **Sequence to One**      | Sequence (e.g., "hell") | One item (e.g., "o") | Sentiment classification, next char prediction |
| 2. **Sequence to Sequence** | Sequence                | Sequence             | Machine translation, text generation           |
| 3. **One to One**           | Single input            | Single output        | Basic classification, regression               |
| 4. **One to Sequence**      | One input               | Sequence             | Image captioning, decoding a fixed embedding   |


In [ ]:
# **Sequence to One**

| Example Task             | Input           | Target Output |
| ------------------------ | --------------- | ------------- |
| Predict next char `"o"`  | `"hell"`        | `"o"`         |
| Sentiment classification | `"i love this"` | `Positive`    |


Use: RNN/GRU/LSTM + fc(last_hidden)  ->    "hell" → o"!

In [ ]:
# Sequence → Sequence (many-to-many)

| Example Task      | Input       | Target Output |
| ----------------- | ----------- | ------------- |
| Language modeling | `"hel"`     | `"ell"`       |
| Translation       | `"je suis"` | `"i am"`      |


Use:

    RNN/GRU/LSTM per timestep
    Often trained with teacher forcing

Variants:

    Equal length: input & output have same length (e.g., "abc" → "bcd")
    Different length: encoder-decoder models (e.g., translation)


In [ ]:
# One → One (simple classification)

| Example Task    | Input           | Output   |
| --------------- | --------------- | -------- |
| Image → Label   | `cat.jpg`       | `"cat"`  |
| Feature → Class | `[age, height]` | `"tall"` |

Used in classic MLPs or CNNs.



In [ ]:
# One → Sequence (image captioning)

| Example Task          | Input             | Output Sequence  |
| --------------------- | ----------------- | ---------------- |
| Image captioning      | `image`           | `"a cat on bed"` |
| Word embedding → Text | `[0.1, 0.3, 0.5]` | `"hello world"`  |


Use:

    Encode input to a fixed vector
    Feed into RNN/GRU/LSTM to generate sequence

In [ ]:
1. Sequence → One
   [w1, w2, w3] → y

2. Sequence → Sequence
   [w1, w2, w3] → [y1, y2, y3]

3. One → One
   x → y

4. One → Sequence
   x → [y1, y2, y3]


In [ ]:
| Task                  | Type                | Model Needed               |
| --------------------- | ------------------- | -------------------------- |
| `"hell"` → `"o"`      | Sequence → One      | GRU + fc                   |
| `"hello"` → `"elloo"` | Sequence → Sequence | GRU (per timestep)         |
| Predict sentiment     | Sequence → One      | GRU + fc                   |
| Predict full word     | Sequence → Sequence | GRU + argmax per time step |
| Caption an image      | One → Sequence      | CNN + GRU decoder          |


In [ ]:
# i can show some examples of translation.....